# CLA in Python 3

Original code in Python 2, together with the example dataset, is here: https://github.com/mdengler/cla

Based on the example by [adalseno](https://github.com/adalseno).

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

import CLA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Python >= 3.9 required

## Helper functions

In [ ]:
def compute_return_risk(
    df: pd.DataFrame, mean: np.ndarray, covar: np.ndarray
) -> tuple[pd.Series, pd.Series]:
    """Compute Return and Risk (volatility) series for the CLA weights

    Args:
        df (pd.DataFrame): Dataframe with the weights
        mean (np.ndarray): Array of mean returns
        covar (np.ndarray): Array of covariances

    Returns:
        tuple[pd.Series, pd.Series]: series of returns and risk
    """
    p_ret = []
    risk = []
    for i in df.index:
        p_ret.append(np.dot(df.loc[i, :].values, mean)[0])
        risk.append(
            np.sqrt(np.dot(df.loc[i, :].values, np.dot(covar, df.loc[i, :].values)))
        )
    return pd.Series(p_ret, index=df.index, name="Return"), pd.Series(
        risk, index=df.index, name="Risk"
    )


def create_weight_table(
    cla: CLA.CLA, mean: np.ndarray, covar: np.ndarray, var_names: list
) -> pd.DataFrame:
    """Create the table with Return, Risk, Lambda and Weights

    Args:
        cla (CLA.CLA): The CLA object with the solution
        mean (np.ndarray): Array with the return means
        covar (np.ndarray): Array of covariances
        var_names (list): List of names for the columns

    Returns:
        pd.DataFrame: The weights table
    """
    weights = pd.DataFrame(
        np.hstack(cla.w).T, columns=var_names, index=np.arange(1, len(cla.w) + 1)
    )
    port_return, port_risk = compute_return_risk(weights, mean, covar)
    port_lambda = pd.Series(cla.l, index=weights.index, name="Lambda")
    return pd.concat([port_return, port_risk, port_lambda, weights], axis=1)


def plot_results(cla: CLA.CLA) -> None:
    """Plots the efficient frontier and Sharpe ratio curve"""
    mu, sigma, weights = cla.efFrontier(100)
    mu = np.array(mu)
    sigma = np.array(sigma)
    x_low = max(sigma.min() - 0.05, 0.0)
    fig, ax = plt.subplots(1, 2, figsize=(20, 7))
    ax[0].plot(sigma, mu, color="blue")
    ax[0].set_xlabel("Risk")
    ax[0].set_ylabel("Expected Excess Return", rotation=90)
    ax[0].set_xlim(x_low, 1.0)
    ax[0].set_title("CLA-derived Efficient Frontier")
    ax[1].plot(sigma, np.round(mu / sigma, 2), color="blue")
    ax[1].set_xlabel("Risk")
    ax[1].set_ylabel("Sharpe ratio", rotation=90)
    ax[1].set_xlim(x_low, 1.0)
    ax[1].set_title("CLA-derived Sharpe Ratio function")
    plt.suptitle("CLA results", fontsize=16)
    plt.show()


def display_results(ret_mean: np.ndarray, cla: CLA.CLA) -> None:
    """Plot the CLA results and display the maximum Sharpe portfolio
    and the minimum variance one"""
    plot_results(cla)
    print()
    # 5) Get Maximum Sharpe ratio portfolio
    sr, w_sr = cla.getMaxSR()
    display(
        Markdown(
            f"### Portfolio volatility: {np.sqrt(np.dot(np.dot(w_sr.ravel(), cla.covar), w_sr.ravel())):.2%}, Sharpe ratio: {sr:.2f}"
        )
    )
    display(Markdown("### Weights (rounded to 4 decimal places):"))
    display(
        pd.Series(
            np.round(w_sr.ravel(), 4),
            name="sr_weights",
            index=np.arange(1, len(w_sr) + 1),
        )
    )
    print()
    # 6) Get Minimum Variance portfolio
    mv, w_mv = cla.getMinVar()
    mu_v = np.dot(w_mv.ravel(), ret_mean)
    sr_v = np.round(mu_v / mv, 2).ravel()[0]
    display(
        Markdown(
            f"### Portfolio minimum volatility: {mv.ravel()[0]:.2%}, Sharpe ratio: {sr_v:.2f}"
        )
    )
    display(Markdown("### Weights (rounded to 4 decimal places):"))
    display(
        pd.Series(
            np.round(w_mv.ravel(), 4),
            name="mv_weights",
            index=np.arange(1, len(w_sr) + 1),
        )
    )
    print()

## Import dataset

In [ ]:
_cols = [f"X{i}" for i in range(1, 11)]
df = pd.read_csv("../CLA_Data.csv")
df

## Create variables

In [ ]:
var_names = df.columns.to_list()
var_names

In [ ]:
mean  = df.iloc[0, :].values.reshape(-1, 1)
lB    = df.iloc[1, :].values.reshape(-1, 1)
uB    = df.iloc[2, :].values.reshape(-1, 1)
covar = df.iloc[3:, :].values

## Compute CLA

In [ ]:
# 3) Invoke object
cla = CLA.CLA(mean, covar, lB, uB)
cla.solve()

## Display results

In [ ]:
weight_df = create_weight_table(cla, mean, covar, var_names)
weight_df.style.format("{:.3f}", na_rep="")

In [ ]:
display_results(mean, cla)